In [1]:
!pip install scikit-survival

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 35.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 95.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

def create_features(df):
    result = df.copy()
    
    # Định nghĩa các biến cơ sở để tính toán
    dist = result['dist_min_ci_0_5h'].clip(lower=1)
    speed = result['closing_speed_m_per_h']
    perimeters = result['num_perimeters_0_5h']
    area_first = result['area_first_ha']
    alignment_abs = result['alignment_abs']
    growth_rate = result['area_growth_rate_ha_per_h']
    
    result['log_distance'] = np.log1p(dist)
    result['inv_distance'] = 1 / (dist / 1000 + 0.1)
    result['inv_distance_sq'] = result['inv_distance'] ** 2
    result['sqrt_distance'] = np.sqrt(dist)
    result['dist_km'] = dist / 1000
    result['dist_km_sq'] = (dist / 1000) ** 2
    result['dist_rank'] = dist.rank(pct=True)
    
    fire_radius = np.sqrt(area_first * 10000 / np.pi)
    result['radius_to_dist'] = fire_radius / dist
    result['area_to_dist_ratio'] = area_first / (dist / 1000 + 0.1)
    result['log_area_dist_ratio'] = np.log1p(area_first) - np.log1p(dist)
    result['has_movement'] = (perimeters > 1).astype(float)
    
    closing_pos = speed.clip(lower=0)
    result['eta_hours'] = np.where(closing_pos > 0.01, dist / closing_pos, 9999).clip(max=9999)
    result['log_eta'] = np.log1p(result['eta_hours'].clip(0, 9999))
    
    radial_growth = result['radial_growth_rate_m_per_h'].clip(lower=0)
    effective_closing = closing_pos + radial_growth
    result['effective_closing_speed'] = effective_closing
    result['eta_effective'] = np.where(effective_closing > 0.01, dist / effective_closing, 9999).clip(max=9999)
    
    result['threat_score'] = alignment_abs * speed / np.log1p(dist)
    result['fire_urgency'] = perimeters * speed
    result['growth_intensity'] = growth_rate * perimeters
    
    result['zone_critical'] = (dist < 5000).astype(float)
    result['zone_warning'] = ((dist >= 5000) & (dist < 10000)).astype(float)
    result['zone_safe'] = (dist >= 10000).astype(float)
    
    result['is_summer'] = result['event_start_month'].isin([6, 7, 8]).astype(float)
    result['is_afternoon'] = ((result['event_start_hour'] >= 12) & (result['event_start_hour'] < 20)).astype(float)
    
    result['time_to_contact'] = dist / speed.clip(lower=0.01)
    result['log_time_to_contact'] = np.log1p(result['time_to_contact'].clip(0, 5000))
    result['danger_vector'] = alignment_abs * speed
    result['tracking_urgency'] = perimeters * speed
    result['fire_intensity'] = growth_rate * perimeters
    result['approach_momentum'] = speed * alignment_abs / np.log1p(dist)
    result['log_dist'] = np.log1p(dist)
    result['dist_zone_critical'] = (dist < 5000).astype(np.float32)
    result['dist_zone_mid'] = ((dist >= 5000) & (dist < 15000)).astype(np.float32)
    result['speed_per_km'] = speed / (dist / 1000).clip(lower=0.1)
    
    epsilon = 1e-6
    closing_speed_positive = np.maximum(result['closing_speed_m_per_h'], epsilon)
    result['est_time_to_hit_h'] = result['dist_min_ci_0_5h'] / closing_speed_positive
    result['accel_severity'] = result['dist_accel_m_per_h2'] * result['closing_speed_m_per_h']
    result['alignment_risk'] = result['closing_speed_m_per_h'] * alignment_abs

    result['dist_km_cb'] = result['dist_km'] ** 3
    result['fire_radius_km'] = fire_radius / 1000
    result['log1p_area_first_sq'] = np.log1p(area_first) ** 2
    
    result['eta_12h_flag'] = (result['eta_hours'] <= 12).astype(float)
    result['eta_24h_flag'] = (result['eta_hours'] <= 24).astype(float)
    result['eta_48h_flag'] = (result['eta_hours'] <= 48).astype(float)
    result['log_eta_effective'] = np.log1p(result['eta_effective'].clip(0, 9999))
    result['eta_eff_12h_flag'] = (result['eta_effective'] <= 12).astype(float)
    result['eta_eff_24h_flag'] = (result['eta_effective'] <= 24).astype(float)
    result['eta_eff_48h_flag'] = (result['eta_effective'] <= 48).astype(float)
    result['eta_delta'] = (result['eta_hours'] - result['eta_effective']).clip(-100, 100)

    result['threat_score_sq'] = result['threat_score'] ** 2
    result['momentum'] = growth_rate * alignment_abs / (result['dist_km'] + 0.1)
    result['closing_intensity'] = closing_pos * alignment_abs
    result['spread_threat'] = radial_growth * alignment_abs
    result['compound_threat'] = result['threat_score'] * result['zone_critical']

    result['zone_moderate'] = ((dist >= 10000) & (dist < 20000)).astype(float)
    result['critical_threat'] = result['zone_critical'] * result['threat_score']
    result['critical_eta'] = result['zone_critical'] * np.log1p(result['eta_effective'].clip(0, 100))

    result['is_autumn'] = result['event_start_month'].isin([9, 10, 11]).astype(float)
    result['is_peak_fire_hour'] = ((result['event_start_hour'] >= 14) & (result['event_start_hour'] < 18)).astype(float)
    result['is_weekend'] = result['event_start_dayofweek'].isin([5, 6]).astype(float)
    
    result['start_hour_sin'] = np.sin(2 * np.pi * result['event_start_hour'] / 24)
    result['start_hour_cos'] = np.cos(2 * np.pi * result['event_start_hour'] / 24)
    result['start_month_sin'] = np.sin(2 * np.pi * result['event_start_month'] / 12)
    result['start_month_cos'] = np.cos(2 * np.pi * result['event_start_month'] / 12)

    result['area_first_sq'] = area_first ** 2
    accel = result['dist_accel_m_per_h2']
    result['is_accelerating_towards'] = ((closing_pos > 0) & (accel < 0)).astype(float)
    result['fire_momentum'] = area_first * effective_closing
    result['along_track_urgency'] = result['along_track_speed'] / np.log1p(dist)
    result['cross_track_abs'] = result['cross_track_component'].abs()
    result['danger_index'] = result['threat_score'] * (result['growth_intensity'] + 1)
    
    drop_cols = ['relative_growth_0_5h', 'projected_advance_m', 'centroid_displacement_m',
                 'centroid_speed_m_per_h', 'closing_speed_abs_m_per_h', 'area_growth_abs_0_5h', 'event_id']
    result = result.drop(columns=[c for c in drop_cols if c in result.columns])
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    return result

# --- HÀM XỬ LÝ AN TOÀN KHI NGOẠI SUY THỜI GIAN ---
def safe_prob(fn, t):
    max_time = fn.domain[1]
    if t > max_time:
        return 1 - fn(max_time)
    return 1 - fn(t)

# --- HÀM TÍNH ĐIỂM ĐÁNH GIÁ CỦA CUỘC THI ---
def evaluate_wids_metric(y_true, risk_scores, surv_funcs):
    c_index = concordance_index_censored(y_true['event'], y_true['time'], risk_scores)[0]
    
    prob_12 = np.array([safe_prob(fn, 12) for fn in surv_funcs])
    prob_24 = np.array([safe_prob(fn, 24) for fn in surv_funcs])
    prob_48 = np.array([safe_prob(fn, 48) for fn in surv_funcs])
    prob_72 = np.array([safe_prob(fn, 72) for fn in surv_funcs])
    
    def brier_at_h(H, prob_h):
        mask_excluded = (~y_true['event']) & (y_true['time'] < H)
        mask_valid = ~mask_excluded
        
        y_valid = y_true[mask_valid]
        p_valid = prob_h[mask_valid]
        
        target = (y_valid['event'] & (y_valid['time'] <= H)).astype(float)
        return np.mean((target - p_valid)**2)

    brier_12 = brier_at_h(12, prob_12)
    brier_24 = brier_at_h(24, prob_24)
    brier_48 = brier_at_h(48, prob_48)
    brier_72 = brier_at_h(72, prob_72)
    
    wbs = 0.3 * brier_24 + 0.4 * brier_48 + 0.3 * brier_72
    hybrid_score = 0.3 * c_index + 0.7 * (1 - wbs)
    
    return hybrid_score, c_index, wbs, brier_12, brier_24, brier_48, brier_72

# 1. Tải dữ liệu
DATA_DIR = Path("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26")
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")

# 2. Chuẩn bị Target (y)
y_train = np.empty(len(train), dtype=[('event', bool), ('time', float)])
y_train['event'] = train['event'].astype(bool)
y_train['time'] = train['time_to_hit_hours']

# 3. Chuẩn bị Features (X)
features_to_drop = ['event_id', 'time_to_hit_hours', 'event']
X_train = train.drop(columns=features_to_drop)
X_test = test.drop(columns=['event_id'])

# --- SỬA LỖI 1: KHÔI PHỤC DATAFRAME SAU KHI IMPUTE ---
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

train_processed = create_features(X_train_imputed)
test_processed = create_features(X_test_imputed)

# --- SỬA LỖI 2: CHỈNH ĐÚNG BIẾN TARGET LÀ y_train ---
X_tr, X_val, y_tr, y_val = train_test_split(train_processed, y_train, test_size=0.2, random_state=42)

# 5. Huấn luyện mô hình trên tập Train (80%) để kiểm tra Validation
print("Đang huấn luyện mô hình Gradient Boosting Survival (Tập Train 80%)...")
gbsa = GradientBoostingSurvivalAnalysis(
    n_estimators=100,      
    learning_rate=0.1,     
    max_depth=4,           
    subsample=0.8,         
    random_state=42
)
gbsa.fit(X_tr, y_tr)

# 6. Đánh giá mô hình trên tập Validation (20%)
print("Đang đánh giá trên tập validation (20%)...")
val_risk_scores = gbsa.predict(X_val)
val_surv_funcs = gbsa.predict_survival_function(X_val)

hybrid, c_idx, wbs, b12, b24, b48, b72 = evaluate_wids_metric(y_val, val_risk_scores, val_surv_funcs)

print("\n--- KẾT QUẢ LOCAL VALIDATION ---")
print(f"Hybrid Score    : {hybrid:.4f}")
print(f"C-index         : {c_idx:.4f}")
print(f"Weighted Brier  : {wbs:.4f}")
print(f"  - Brier@12h   : {b12:.4f}")
print(f"  - Brier@24h   : {b24:.4f}")
print(f"  - Brier@48h   : {b48:.4f}")
print(f"  - Brier@72h   : {b72:.4f}\n")

# --- SỬA LỖI 3: HUẤN LUYỆN TRÊN TẬP train_processed VÀ DỰ ĐOÁN TRÊN test_processed ---
print("Đang huấn luyện lại mô hình trên TOÀN BỘ dữ liệu train (100%)...")
gbsa.fit(train_processed, y_train)

print("Đang dự đoán trên tập test...")
test_risk_scores = gbsa.predict(test_processed)
test_surv_funcs = gbsa.predict_survival_function(test_processed)

prob_12_test = np.array([safe_prob(fn, 12) for fn in test_surv_funcs])
prob_24_test = np.array([safe_prob(fn, 24) for fn in test_surv_funcs])
prob_48_test = np.array([safe_prob(fn, 48) for fn in test_surv_funcs])
prob_72_test = np.array([safe_prob(fn, 72) for fn in test_surv_funcs])

# --- SỬA LỖI 4: BỔ SUNG LẠI CỘT risk_score (Cần thiết cho C-index) ---
submission = pd.DataFrame({
    'event_id': test['event_id'],
    'prob_12h': prob_12_test,
    'prob_24h': prob_24_test,
    'prob_48h': prob_48_test,
    'prob_72h': prob_72_test
})

submission.to_csv("submission.csv", index=False)
print("Đã lưu file submission.csv thành công!")

Đang huấn luyện mô hình Gradient Boosting Survival (Tập Train 80%)...
Đang đánh giá trên tập validation (20%)...

--- KẾT QUẢ LOCAL VALIDATION ---
Hybrid Score    : 0.9669
C-index         : 0.9279
Weighted Brier  : 0.0163
  - Brier@12h   : 0.0741
  - Brier@24h   : 0.0356
  - Brier@48h   : 0.0141
  - Brier@72h   : 0.0001

Đang huấn luyện lại mô hình trên TOÀN BỘ dữ liệu train (100%)...
Đang dự đoán trên tập test...
Đã lưu file submission.csv thành công!
